In [1]:
from importlib import reload
import joblib
import matplotlib.pyplot as plt
import numpy as np
import scipy

import pycrp
reload(pycrp)

import sys

sys.path.append("../")

import _config as config
import helper_basic as hp
reload(config)

<module '_config' from '/home/tomren/repo/CorrFitV2/fit/../_config.py'>

In [ ]:
data = joblib.load("../data/trace_signal/combined_traces.joblib")

In [3]:
masks = joblib.load("../data/final_psd_and_templates/selected_traces_masks.joblib")
mask_good = masks["mask_chi2"]

In [2]:
CHANNELS = config.chs # list(data.keys())
LEN_TRACE = 32768 # len(data[CHANNELS[0]][0])
LEN_PRE = 16000
TEMPLATE_T0 = 16035
FS = 625_000
dt = 1/FS

adc2amp = 8.192/ 65536 / 5000 / 2.4 /2
FFT_FREQ = np.fft.rfftfreq(LEN_TRACE, dt)

csds = joblib.load("../data/final_psd_and_templates/psd_csd_combined_22800avg.joblib")

all_psds = [csds[f"{ch}_{ch}"] / LEN_TRACE for ch in CHANNELS]
total_psd = np.zeros_like(all_psds[0])
for ch in CHANNELS:
    for ch2 in CHANNELS:
        try:
            total_psd += csds[f"{ch}_{ch2}"].real / LEN_TRACE
        except:
            total_psd += csds[f"{ch2}_{ch}"].real / LEN_TRACE

LIMIT_TIME_OF = 150


templates_pca =  joblib.load("../data/pca/templates_pca_543.joblib")
all_templates = [templates_pca["template_avg"][ch] for ch in CHANNELS]


iOFALL = pycrp.filter.OFALL(all_templates, all_psds, freq_limit=-1, window=None, lowfreq_chi2_bins=100)
iOFs = {}
for ich, ch in enumerate(CHANNELS):
    iOFs[ch] = pycrp.filter.OF(all_templates[ich], all_psds[ich], fs = 1)
OF_Total = pycrp.filter.OF(np.mean(all_templates, axis=0), np.sum(all_psds, axis=0), fs = 1, lowfreq_chi2_bins =100)


In [ ]:
templates_pca["variance_pca"]

{'PFS1': array([1.19257958, 0.05888726, 0.04905037]),
 'PCS1': array([4.28713768, 0.22983975, 0.20267049]),
 'PDS1': array([3.59802638, 0.11672102, 0.01209764]),
 'PBS1': array([2.33577552, 0.11456384, 0.02860823]),
 'PES1': array([4.06405479, 0.10779968, 0.00934683]),
 'PAS1': array([2.28438335, 0.11285957, 0.030474  ]),
 'PFS2': array([1.31156584, 0.03750679, 0.02189748]),
 'PCS2': array([4.0132845 , 0.13564793, 0.01200574]),
 'PBS2': array([1.77316722, 0.21147936, 0.02683715]),
 'PES2': array([3.93865706, 0.13301249, 0.01434476]),
 'PDS2': array([4.41294683, 0.15729166, 0.03941024]),
 'PAS2': array([1.88336741, 0.12942908, 0.02059207])}